### Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

### Functions and utilities

In [ ]:
np.random.seed(42)

In [ ]:
#reads a file. Each line has the format: label text
#Returns a list with the text and a list with the labels
def readData(fname):

    with open(fname, 'r', encoding="utf-8") as f:
        fileData = f.read()
  
    lines = fileData.split("\n")
    textData = list()
    textLabel = list()
    lineLength = np.zeros(len(lines))
    
    for i, aLine in enumerate(lines):     
        if not aLine:
            break  
        label = aLine.split(" ")[0]
        lineLength[i] = len(aLine.split(" "))
        if(label == "__label__1"):
            textLabel.append(0)
            textData.append(aLine.removeprefix("__label__1 "))

        elif(label == "__label__2"):
            textLabel.append(1)
            textData.append(aLine.removeprefix("__label__2 "))

        else:
            print("\nError in readData: ", i, aLine)
            exit()
    
    f.close()
    return textData, textLabel, int(np.average(lineLength)+2*np.std(lineLength))

In [ ]:
from tensorflow.keras.layers import TextVectorization

def transformData(x_train, y_train, x_test, y_test, maxFeatures, seqLength):
    #transforms text input to int input based on the vocabulary
    #max_tokens = maxFeatures is the size of the vocabulary
    #output_sequence_length =  seqLength is the maximum length of the transformed text. Adds 0 is text length is shorter
    precLayer = TextVectorization(max_tokens = maxFeatures, 
    standardize =  'lower_and_strip_punctuation', split = 'whitespace', output_mode = 'int', 
    output_sequence_length =  seqLength)
    precLayer.adapt(x_train)
    #print(precLayer.get_vocabulary())
    x_train_int = precLayer(x_train)
    y_train = tf.convert_to_tensor(y_train)
    #print(x_train_int)
    #print(y_train)
    x_test_int= precLayer(x_test)
    y_test = tf.convert_to_tensor(y_test)
    #print(x_test_int)
    #print(y_test)

    return x_train_int, y_train, x_test_int, y_test

### Loading the data

In [ ]:
x_train, y_train, seqLength = readData("amazon/train_small.txt")
x_test, y_test, tmp = readData("amazon/test_small.txt")

#maxFeatures is a hyperparameter
maxFeatures = 10000

x_train_int, y_train, x_test_int, y_test = transformData(x_train, y_train, x_test, y_test, maxFeatures, seqLength)

### Creating the models

**MODEL 1**

In [ ]:
model1 = keras.Sequential()
model1.add(layers.Input(shape=(seqLength,)))
model1.add(layers.Embedding(input_dim=maxFeatures, output_dim=128, input_length=seqLength))
model1.add(layers.Bidirectional(layers.LSTM(128)))
model1.add(layers.Dropout(0.5))
model1.add(layers.Dense(1, activation='sigmoid'))

model1.summary()

In [ ]:
model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

**MODEL 2**

In [ ]:
model2 = keras.Sequential()
model2.add(layers.Input(shape=(seqLength,)))
model2.add(layers.Embedding(input_dim=maxFeatures, output_dim=128, input_length=seqLength))
model2.add(layers.Bidirectional(layers.LSTM(16, return_sequences=True)))
model2.add(layers.Bidirectional(layers.LSTM(16, return_sequences=True)))
model2.add(layers.Bidirectional(layers.LSTM(16)))
model2.add(layers.Dropout(0.5))
model2.add(layers.Dense(1, activation='sigmoid'))

model2.summary()

In [ ]:
model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

**MODEL 3**

In [ ]:
model3 = keras.Sequential()
model3.add(layers.Input(shape=(seqLength,)))
model3.add(layers.Embedding(input_dim=maxFeatures, output_dim=128, input_length=seqLength))
model3.add(layers.Bidirectional(layers.GRU(128)))
model3.add(layers.Dropout(0.5))
model3.add(layers.Dense(1, activation='sigmoid'))

model3.summary()

In [ ]:
model3.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

**MODEL 4**

In [ ]:
# Ruben's model
model4 = keras.Sequential()
model4.add(layers.Input(shape=(seqLength,)))
model4.add(layers.Embedding(input_dim=maxFeatures, output_dim=128, input_length=seqLength))
model4.add(layers.GRU(128, activation='tanh', return_sequences=True))
model4.add(layers.LSTM(16, return_sequences=True))
model4.add(layers.GRU(16))
model4.add(layers.Dense(1, activation='sigmoid'))

model4.summary()

In [ ]:
model4.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

**MODEL 5**

In [ ]:
model5 = keras.Sequential()
model5.add(layers.Input(shape=(seqLength,)))
model5.add(layers.Embedding(input_dim=maxFeatures, output_dim=128, input_length=seqLength))
model5.add(layers.Conv1D(128, 3, activation='relu'))
model5.add(layers.MaxPooling1D(2))
model5.add(layers.Conv1D(128, 3, activation='relu'))
model5.add(layers.MaxPooling1D(2))
model5.add(layers.Bidirectional(layers.LSTM(128)))
model5.add(layers.Dropout(0.5))
model5.add(layers.Dense(1, activation='sigmoid'))

model5.summary()

In [ ]:
model5.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

### Training the models

In [ ]:
# Creating early stopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

**MODEL 1**

In [ ]:
history1 = model1.fit(x_train_int, y_train, epochs=10, batch_size=64, validation_split=0.2, callbacks=[early_stopping])

plt.plot(history1.history['accuracy'], label='accuracy')
plt.plot(history1.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
model1.save('model_1.h5')

In [ ]:
history_df = pd.DataFrame(history1.history)
history_df.to_csv('training_history_model_1.csv', index=False)

**MODEL 2**

In [ ]:
history2 = model2.fit(x_train_int, y_train, epochs=10, batch_size=64, validation_split=0.2, callbacks=[early_stopping])

plt.plot(history2.history['accuracy'], label='accuracy')
plt.plot(history2.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
model2.save('model_2.h5')

In [ ]:
history_df = pd.DataFrame(history2.history)
history_df.to_csv('training_history_model_2.csv', index=False)

**MODEL 3**

In [ ]:
history3 = model3.fit(x_train_int, y_train, epochs=10, batch_size=32, validation_split=0.2, callbacks=[early_stopping])

plt.plot(history3.history['accuracy'], label='accuracy')
plt.plot(history3.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
model3.save('model_3.h5')

In [ ]:
history_df = pd.DataFrame(history3.history)
history_df.to_csv('training_history_model_3.csv', index=False)

**MODEL 4**

In [ ]:
history4 = model4.fit(x_train_int, y_train, epochs=10, batch_size=128, validation_split=0.2, callbacks=[early_stopping])

plt.plot(history4.history['accuracy'], label='accuracy')
plt.plot(history4.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
model4.save('model_4.h5')

In [ ]:
history_df = pd.DataFrame(history4.history)
history_df.to_csv('training_history_model_4.csv', index=False)

**MODEL 5**

In [ ]:
history5 = model5.fit(x_train_int, y_train, epochs=10, batch_size=64, validation_split=0.2, callbacks=[early_stopping])

plt.plot(history5.history['accuracy'], label='accuracy')
plt.plot(history5.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
model5.save('model_5.h5')

In [ ]:
history_df = pd.DataFrame(history5.history)
history_df.to_csv('training_history_model_5.csv', index=False)

### Evaluating the models

**MODEL 1**

In [ ]:
loss, accuracy = model1.evaluate(x_test_int, y_test)
print(f"Test accuracy: {accuracy:.4f}")

**MODEL 2**

In [ ]:
loss, accuracy = model2.evaluate(x_test_int, y_test)
print(f"Test accuracy: {accuracy:.4f}")

**MODEL 3**

In [ ]:
loss, accuracy = model3.evaluate(x_test_int, y_test)
print(f"Test accuracy: {accuracy:.4f}")

**MODEL 4**

In [ ]:
loss, accuracy = model4.evaluate(x_test_int, y_test)
print(f"Test accuracy: {accuracy:.4f}")

**MODEL 5**

In [ ]:
loss, accuracy = model5.evaluate(x_test_int, y_test)
print(f"Test accuracy: {accuracy:.4f}")